In [0]:
"""Generic chronological splitting utilities for time-series data."""

from __future__ import annotations

from math import floor


def temporal_split(
    df,
    time_col: str,
    train_fraction: float = 0.70,
    validation_fraction: float = 0.15,
):
    """
    Split a Spark DataFrame chronologically using distinct timestamps.

    The split is based on the ordered unique values of `time_col`, not on
    row counts. All rows sharing the same timestamp are therefore assigned
    to the same partition.

    Args:
        df:
            Input Spark DataFrame.
        time_col:
            Name of the timestamp/date column.
        train_fraction:
            Fraction of distinct timestamps assigned to training.
        validation_fraction:
            Fraction of distinct timestamps assigned to validation.

    Returns:
        Tuple containing:
            - training DataFrame
            - validation DataFrame
            - test DataFrame
            - training end timestamp
            - validation end timestamp

    Raises:
        ValueError:
            If the time column does not exist, split fractions are invalid,
            or there are too few distinct timestamps to create all splits.
    """
    if time_col not in df.columns:
        raise ValueError(f"Time column '{time_col}' does not exist in the DataFrame.")

    if not 0 < train_fraction < 1:
        raise ValueError("train_fraction must be between 0 and 1.")

    if not 0 < validation_fraction < 1:
        raise ValueError("validation_fraction must be between 0 and 1.")

    if train_fraction + validation_fraction >= 1:
        raise ValueError("train_fraction + validation_fraction must be less than 1.")

    timestamps = [
        row[time_col]
        for row in (
            df.select(time_col)
            .where(df[time_col].isNotNull())
            .distinct()
            .orderBy(time_col)
            .collect()
        )
    ]

    n_timestamps = len(timestamps)

    if n_timestamps < 3:
        raise ValueError("At least 3 distinct non-null timestamps are required.")

    n_train = floor(n_timestamps * train_fraction)

    n_validation = floor(n_timestamps * validation_fraction)

    n_test = n_timestamps - n_train - n_validation

    if n_train < 1 or n_validation < 1 or n_test < 1:
        raise ValueError(
            "Split fractions produce an empty train, validation, or test split."
        )

    train_end = timestamps[n_train - 1]

    validation_end = timestamps[n_train + n_validation - 1]

    train_df = df.filter(df[time_col] <= train_end)

    validation_df = df.filter(
        (df[time_col] > train_end) & (df[time_col] <= validation_end)
    )

    test_df = df.filter(df[time_col] > validation_end)

    return (
        train_df,
        validation_df,
        test_df,
        train_end,
        validation_end,
    )


In [0]:
gold_df = spark.table(
    "finance_ml.gold.account_features"
)

(
    train_df,
    validation_df,
    test_df,
    train_end,
    validation_end,
) = temporal_split(
    gold_df,
    time_col="date",
    train_fraction=0.70,
    validation_fraction=0.15,
)

print("train_end:", train_end)
print("validation_end:", validation_end)

print("train rows:", train_df.count())
print("validation rows:", validation_df.count())
print("test rows:", test_df.count())

In [0]:
def get_model_columns(
    df,
    target_col: str = "risk_state",
    target_name_col: str = "risk_state_name",
    id_columns: tuple[str, ...] = ("account_id", "date"),
    categorical_columns: tuple[str, ...] = ("account_type",),
):
    """
    Identify model feature, target, identifier, and categorical columns.

    Args:
        df:
            Input Spark DataFrame.
        target_col:
            Numeric target column used for supervised learning.
        target_name_col:
            Human-readable target label column.
        id_columns:
            Columns retained for traceability but excluded from model inputs.
        categorical_columns:
            Feature columns requiring explicit encoding before modeling.

    Returns:
        Dictionary containing:
            - feature_columns
            - numeric_feature_columns
            - categorical_feature_columns
            - target_column
            - target_name_column
            - id_columns

    Raises:
        ValueError:
            If any required column is missing from the DataFrame.
    """
    required_columns = {
        target_col,
        target_name_col,
        *id_columns,
        *categorical_columns,
    }

    missing_columns = sorted(required_columns - set(df.columns))

    if missing_columns:
        raise ValueError("Missing required columns: " + ", ".join(missing_columns))

    excluded_columns = {
        target_col,
        target_name_col,
        *id_columns,
    }

    feature_columns = [
        column for column in df.columns if column not in excluded_columns
    ]

    categorical_feature_columns = [
        column for column in categorical_columns if column in feature_columns
    ]

    numeric_feature_columns = [
        column
        for column in feature_columns
        if column not in categorical_feature_columns
    ]

    return {
        "feature_columns": feature_columns,
        "numeric_feature_columns": numeric_feature_columns,
        "categorical_feature_columns": categorical_feature_columns,
        "target_column": target_col,
        "target_name_column": target_name_col,
        "id_columns": list(id_columns),
    }


In [0]:
model_columns = get_model_columns(gold_df)

print("Feature columns:")
for column in model_columns["feature_columns"]:
    print("-", column)

print("\nNumeric features:")
for column in model_columns["numeric_feature_columns"]:
    print("-", column)

print("\nCategorical features:")
for column in model_columns["categorical_feature_columns"]:
    print("-", column)

In [0]:
from pyspark.sql import functions as F


def missing_counts(df, columns):
    return df.select(
        *[
            F.sum(
                F.col(column).isNull().cast("int")
            ).alias(column)
            for column in columns
        ]
    )


numeric_features = model_columns[
    "numeric_feature_columns"
]

print("Train missingness")
display(
    missing_counts(
        train_df,
        numeric_features,
    )
)

print("Validation missingness")
display(
    missing_counts(
        validation_df,
        numeric_features,
    )
)

print("Test missingness")
display(
    missing_counts(
        test_df,
        numeric_features,
    )
)

In [0]:
"""Leakage-safe numeric imputation utilities for model datasets."""

from __future__ import annotations


def fit_numeric_medians(
    train_df,
    numeric_columns: list[str],
) -> dict[str, float]:
    """
    Fit median imputation values using training data only.

    Args:
        train_df:
            Training Spark DataFrame.
        numeric_columns:
            Numeric feature columns to inspect.

    Returns:
        Dictionary mapping columns with missing values to their
        training-derived median.
    """
    from pyspark.sql import functions as F

    missing_columns = []

    for column in numeric_columns:
        has_missing = train_df.filter(F.col(column).isNull()).limit(1).count() > 0

        if has_missing:
            missing_columns.append(column)

    medians = {}

    for column in missing_columns:
        median_value = train_df.select(
            F.percentile_approx(
                column,
                0.5,
            ).alias("median")
        ).first()["median"]

        if median_value is None:
            raise ValueError(
                f"Cannot compute median for '{column}' because "
                "the training column contains no non-null values."
            )

        medians[column] = float(median_value)

    return medians


def apply_numeric_imputation(
    df,
    medians: dict[str, float],
):
    """
    Apply fitted medians and create missing-indicator features.

    Args:
        df:
            Spark DataFrame to transform.
        medians:
            Training-derived median values keyed by feature name.

    Returns:
        Spark DataFrame containing:
            - original numeric columns with missing values imputed
            - `<column>_missing` indicator columns
    """
    from pyspark.sql import functions as F

    result_df = df

    for column, median_value in medians.items():
        missing_indicator = f"{column}_missing"

        result_df = result_df.withColumn(
            missing_indicator,
            F.col(column).isNull().cast("int"),
        ).withColumn(
            column,
            F.coalesce(
                F.col(column),
                F.lit(median_value),
            ),
        )

    return result_df


In [0]:
# Inspect the medians from train split (to be used for imputation)

numeric_features = model_columns[
    "numeric_feature_columns"
]

medians = fit_numeric_medians(
    train_df,
    numeric_features,
)

print("Learned training medians:")
for column, value in medians.items():
    print(f"{column}: {value}")

In [0]:
# Transform all three splits with the same fitted medians:
train_imputed_df = apply_numeric_imputation(
    train_df,
    medians,
)

validation_imputed_df = apply_numeric_imputation(
    validation_df,
    medians,
)

test_imputed_df = apply_numeric_imputation(
    test_df,
    medians,
)

In [0]:
# Verify that the original numeric columns no longer contain nulls:
print("Train missingness after imputation")
display(
    missing_counts(
        train_imputed_df,
        numeric_features,
    )
)

print("Validation missingness after imputation")
display(
    missing_counts(
        validation_imputed_df,
        numeric_features,
    )
)

print("Test missingness after imputation")
display(
    missing_counts(
        test_imputed_df,
        numeric_features,
    )
)

In [0]:
"""Leakage-safe categorical encoding (dummy variable / one-hot encoding) utilities for model datasets."""

from collections.abc import Iterable


def fit_category_levels(
    train_df,
    categorical_columns: Iterable[str],
) -> dict[str, list[str]]:
    """
    Learn categorical levels from training data only.

    Args:
        train_df:
            Training Spark DataFrame.
        categorical_columns:
            Categorical feature columns to encode.

    Returns:
        Dictionary mapping each categorical column to its sorted
        training-derived category levels.

    Raises:
        ValueError:
            If a requested categorical column is missing or contains
            no non-null category values.
    """
    from pyspark.sql import functions as F

    category_levels: dict[str, list[str]] = {}

    for column in categorical_columns:
        if column not in train_df.columns:
            raise ValueError(
                f"Categorical column '{column}' does not exist in the DataFrame."
            )

        levels = [
            row[column]
            for row in (
                train_df.select(column)
                .where(F.col(column).isNotNull())
                .distinct()
                .orderBy(column)
                .collect()
            )
        ]

        if not levels:
            raise ValueError(
                f"Categorical column '{column}' contains no non-null values."
            )

        category_levels[column] = levels

    return category_levels


def apply_one_hot_encoding(
    df,
    category_levels: dict[str, list[str]],
):
    """
    Apply one-hot encoding using training-derived category levels.

    Args:
        df:
            Spark DataFrame to transform.
        category_levels:
            Training-derived category levels for each categorical column.

    Returns:
        Spark DataFrame containing one-hot encoded indicator columns.

    Notes:
        Categories not observed during training produce zeros across all
        one-hot columns for that categorical feature.
    """
    from pyspark.sql import functions as F

    result_df = df

    for column, levels in category_levels.items():
        if column not in result_df.columns:
            raise ValueError(
                f"Categorical column '{column}' does not exist in the DataFrame."
            )

        for level in levels:
            safe_level = str(level).strip().lower().replace(" ", "_").replace("-", "_")

            encoded_column = f"{column}_{safe_level}"

            result_df = result_df.withColumn(
                encoded_column,
                (F.col(column) == F.lit(level)).cast("int"),
            )

    return result_df


In [0]:
categorical_features = model_columns[
    "categorical_feature_columns"
]

category_levels = fit_category_levels(
    train_imputed_df,
    categorical_features,
)

print("Learned category levels:")
for column, levels in category_levels.items():
    print(f"{column}: {levels}")

In [0]:
train_encoded_df = apply_one_hot_encoding(
    train_imputed_df,
    category_levels,
)

validation_encoded_df = apply_one_hot_encoding(
    validation_imputed_df,
    category_levels,
)

test_encoded_df = apply_one_hot_encoding(
    test_imputed_df,
    category_levels,
)

In [0]:
print(
    [
        column
        for column in train_encoded_df.columns
        if column.startswith("account_type_")
    ]
)

In [0]:
"""Utilities for constructing the final numerical model feature schema."""

def build_final_feature_columns(
    numeric_feature_columns: list[str],
    medians: dict[str, float],
    category_levels: dict[str, list[str]],
) -> list[str]:
    """
    Build the final numerical feature-column list used by the model.

    The final schema contains:

    - original numeric features
    - missing-indicator columns for numerics that were imputed
    - one-hot encoded categorical features

    Args:
        numeric_feature_columns:
            Original numerical model features.
        medians:
            Training-derived medians. Each key identifies a numeric feature
            for which a missing-indicator column was created.
        category_levels:
            Training-derived categorical levels used for one-hot encoding.

    Returns:
        Ordered list of final numerical feature-column names.
    """
    final_columns = list(numeric_feature_columns)

    missing_indicator_columns = [f"{column}_missing" for column in medians]

    final_columns.extend(missing_indicator_columns)

    for column, levels in category_levels.items():
        for level in levels:
            safe_level = str(level).strip().lower().replace(" ", "_").replace("-", "_")

            final_columns.append(f"{column}_{safe_level}")

    return final_columns


In [0]:
final_feature_columns = build_final_feature_columns(
    numeric_features,
    medians,
    category_levels,
)

print("Final feature count:", len(final_feature_columns))

print("\nFinal feature columns:")
for column in final_feature_columns:
    print("-", column)

In [0]:
# Verify those columns exist in all three transformed splits:
for name, df in [
    ("train", train_encoded_df),
    ("validation", validation_encoded_df),
    ("test", test_encoded_df),
]:
    missing = [
        column
        for column in final_feature_columns
        if column not in df.columns
    ]

    print(
        f"{name}:",
        "OK" if not missing else missing,
    )

In [0]:
"""Leakage-safe feature scaling utilities for model datasets (only for the continuous/original numeric features)."""

def fit_standard_scaler(
    train_df,
    columns: list[str],
) -> dict[str, dict[str, float]]:
    """
    Fit mean and standard deviation using training data only.

    Args:
        train_df:
            Training Spark DataFrame.
        columns:
            Continuous numeric columns to standardize.

    Returns:
        Dictionary mapping each column to its training-derived
        mean and standard deviation.

    Raises:
        ValueError:
            If a requested column is missing or has zero standard deviation.
    """
    from pyspark.sql import functions as F

    for column in columns:
        if column not in train_df.columns:
            raise ValueError(
                f"Scaling column '{column}' does not exist in the DataFrame."
            )

    aggregations = []

    for column in columns:
        aggregations.extend(
            [
                F.avg(column).alias(f"{column}__mean"),
                F.stddev_pop(column).alias(f"{column}__std"),
            ]
        )

    statistics_row = train_df.select(*aggregations).first()

    scaler_parameters: dict[
        str,
        dict[str, float],
    ] = {}

    for column in columns:
        mean_value = statistics_row[f"{column}__mean"]

        std_value = statistics_row[f"{column}__std"]

        if mean_value is None or std_value is None:
            raise ValueError(f"Cannot fit scaler for '{column}'.")

        if std_value == 0:
            raise ValueError(
                f"Cannot standardize '{column}' because "
                "its training standard deviation is zero."
            )

        scaler_parameters[column] = {
            "mean": float(mean_value),
            "std": float(std_value),
        }

    return scaler_parameters


def apply_standard_scaling(
    df,
    scaler_parameters: dict[
        str,
        dict[str, float],
    ],
):
    """
    Apply training-derived standardization parameters.

    Args:
        df:
            Spark DataFrame to transform.
        scaler_parameters:
            Training-derived mean and standard deviation for each feature.

    Returns:
        Spark DataFrame with the requested columns standardized.
    """
    from pyspark.sql import functions as F

    result_df = df

    for column, parameters in scaler_parameters.items():
        if column not in result_df.columns:
            raise ValueError(
                f"Scaling column '{column}' does not exist in the DataFrame."
            )

        mean_value = parameters["mean"]

        std_value = parameters["std"]

        result_df = result_df.withColumn(
            column,
            (F.col(column) - F.lit(mean_value)) / F.lit(std_value),
        )

    return result_df


In [0]:
scaling_columns = numeric_features

scaler_parameters = fit_standard_scaler(
    train_encoded_df,
    scaling_columns,
)

train_scaled_df = apply_standard_scaling(
    train_encoded_df,
    scaler_parameters,
)

validation_scaled_df = apply_standard_scaling(
    validation_encoded_df,
    scaler_parameters,
)

test_scaled_df = apply_standard_scaling(
    test_encoded_df,
    scaler_parameters,
)

In [0]:
# Verify the training statistics for scaled columns
# We do not expect validation/test to have exactly mean 0 and std 1, because they are transformed using the training statistics, not their own. That is the correct leakage-safe behavior.

from pyspark.sql import functions as F

check_columns = [
    "balance",
    "cash_outflow",
    "payment_ratio",
]

train_scaled_df.select(
    *[
        F.avg(column).alias(
            f"{column}_mean"
        )
        for column in check_columns
    ],
    *[
        F.stddev_pop(column).alias(
            f"{column}_std"
        )
        for column in check_columns
    ],
).show(truncate=False)

In [0]:
# create the final model-ready DataFrames by selecting only:
# identifiers needed for sequence construction,
# final numerical feature columns,
# target columns.

model_train_df = train_scaled_df.select(
    "account_id",
    "date",
    *final_feature_columns,
    "risk_state",
    "risk_state_name",
)

model_validation_df = validation_scaled_df.select(
    "account_id",
    "date",
    *final_feature_columns,
    "risk_state",
    "risk_state_name",
)

model_test_df = test_scaled_df.select(
    "account_id",
    "date",
    *final_feature_columns,
    "risk_state",
    "risk_state_name",
)

In [0]:
# Verify shapes

print("train rows:", model_train_df.count())
print("validation rows:", model_validation_df.count())
print("test rows:", model_test_df.count())

print("model columns:", len(model_train_df.columns))
print("feature count:", len(final_feature_columns))

In [0]:
"""Sequence-construction utilities for time-series classification."""

from __future__ import annotations

from collections.abc import Iterable

import numpy as np


def build_sequences(
    df,
    feature_columns: Iterable[str],
    target_col: str,
    group_col: str,
    time_col: str,
    sequence_length: int,
    stride: int = 1,
):
    """
    Build fixed-length sequence-to-sequence samples from ordered panel data.

    Each returned sample contains:

        X: shape (sequence_length, n_features)
        y: shape (sequence_length,)

    Sequences are created independently inside each `group_col` value and
    therefore never cross entity boundaries.

    Args:
        df:
            Spark DataFrame containing model-ready features and targets.
        feature_columns:
            Numerical feature columns used as model inputs.
        target_col:
            Target column containing one label per timestep.
        group_col:
            Entity identifier used to separate independent time series.
        time_col:
            Timestamp/date column used for chronological ordering.
        sequence_length:
            Number of timesteps per sequence.
        stride:
            Number of rows to advance between consecutive windows.

    Returns:
        Tuple containing:
            - X as a NumPy array with shape
              (n_sequences, sequence_length, n_features)
            - y as a NumPy array with shape
              (n_sequences, sequence_length)
            - metadata list containing group and time boundaries

    Raises:
        ValueError:
            If required columns are missing or sequence parameters are invalid.
    """
    feature_columns = list(feature_columns)

    required_columns = {
        *feature_columns,
        target_col,
        group_col,
        time_col,
    }

    missing_columns = sorted(required_columns - set(df.columns))

    if missing_columns:
        raise ValueError("Missing required columns: " + ", ".join(missing_columns))

    if sequence_length < 1:
        raise ValueError("sequence_length must be at least 1.")

    if stride < 1:
        raise ValueError("stride must be at least 1.")

    selected_df = df.select(
        group_col,
        time_col,
        *feature_columns,
        target_col,
    ).orderBy(
        group_col,
        time_col,
    )

    rows = selected_df.collect()

    grouped_rows: dict[object, list] = {}

    for row in rows:
        group_value = row[group_col]

        grouped_rows.setdefault(
            group_value,
            [],
        ).append(row)

    X_sequences = []
    y_sequences = []
    metadata = []

    for group_value, group_rows in grouped_rows.items():
        n_rows = len(group_rows)

        if n_rows < sequence_length:
            continue

        for start_index in range(
            0,
            n_rows - sequence_length + 1,
            stride,
        ):
            end_index = start_index + sequence_length

            window_rows = group_rows[start_index:end_index]

            X_window = [
                [float(row[column]) for column in feature_columns]
                for row in window_rows
            ]

            y_window = [int(row[target_col]) for row in window_rows]

            X_sequences.append(X_window)

            y_sequences.append(y_window)

            metadata.append(
                {
                    "group": group_value,
                    "start_time": window_rows[0][time_col],
                    "end_time": window_rows[-1][time_col],
                }
            )

    X = np.asarray(
        X_sequences,
        dtype=np.float32,
    )

    y = np.asarray(
        y_sequences,
        dtype=np.int64,
    )

    return X, y, metadata


In [0]:
sequence_length = 7
stride = 1

In [0]:
X_train, y_train, train_metadata = build_sequences(
    model_train_df,
    feature_columns=final_feature_columns,
    target_col="risk_state",
    group_col="account_id",
    time_col="date",
    sequence_length=sequence_length,
    stride=stride,
)

X_validation, y_validation, validation_metadata = build_sequences(
    model_validation_df,
    feature_columns=final_feature_columns,
    target_col="risk_state",
    group_col="account_id",
    time_col="date",
    sequence_length=sequence_length,
    stride=stride,
)

X_test, y_test, test_metadata = build_sequences(
    model_test_df,
    feature_columns=final_feature_columns,
    target_col="risk_state",
    group_col="account_id",
    time_col="date",
    sequence_length=sequence_length,
    stride=stride,
)

In [0]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)